# Unidade IV — Mineração de Padrões

## Avaliação de regras de associação

**Carga estimada:** 2 horas  
**Pré-requisitos:** suporte, confiança, *lift* e regras de associação.

> **Pergunta norteadora:** quando uma regra forte merece atenção e como interpretar suas métricas sem confundir associação com causalidade?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- distinguir regra frequente, forte, interessante e acionável;
- interpretar *lift*, alavancagem e convicção;
- reconhecer redundância, taxa-base e associação sem causalidade;
- relacionar fórmulas, contagens e interpretação das métricas.


In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder


## *Leverage* e *conviction*: avaliando além do suporte e da confiança

Em regras de associação, como as utilizadas pelo algoritmo Apriori, *leverage* (alavancagem) e *conviction* (convicção) são métricas usadas para avaliar a força e a relevância de uma regra do tipo

$$A\rightarrow B,$$

ou seja:

> **Se $A$ ocorre, então $B$ tende a ocorrer.**

Enquanto **suporte** e **confiança** indicam, respectivamente, a frequência da regra e a proporção de ocorrências de $A$ acompanhadas por $B$, *leverage* e *conviction* ajudam a avaliar se existe uma associação efetiva entre $A$ e $B$. Para isso, comparam o comportamento observado com o que seria esperado caso os itens fossem independentes.

Uma regra pode superar os limiares mínimos de suporte e confiança e ainda ser óbvia, redundante, instável ou pouco útil. As medidas a seguir acrescentam duas perspectivas: o excesso de coocorrências e a quantidade de violações da regra.


In [2]:
transacoes = [
    ["arroz", "feijao", "oleo"], ["arroz", "feijao"],
    ["arroz", "leite"], ["pao", "leite", "manteiga"],
    ["pao", "leite"], ["arroz", "feijao", "oleo"],
    ["pao", "cafe"], ["arroz", "feijao", "leite"],
    ["pao", "leite", "manteiga"], ["arroz", "feijao"],
    ["cafe", "leite"], ["arroz", "feijao", "oleo", "leite"],
]
te = TransactionEncoder()
cestas = pd.DataFrame(te.fit(transacoes).transform(transacoes), columns=te.columns_)
itemsets = apriori(cestas, min_support=0.15, use_colnames=True)
regras = association_rules(itemsets, metric="confidence", min_threshold=0.60)
colunas = ["antecedents", "consequents", "support", "confidence", "lift", "leverage", "conviction"]
regras[colunas].sort_values(["lift", "support"], ascending=False).head(12).round(3)


,antecedents,consequents,support,confidence,lift,leverage,conviction
13,"(leite, pao)",(manteiga),0.167,0.667,4.000,0.125,2.5
15,(manteiga),"(leite, pao)",0.167,1.000,4.000,0.125,inf
6,(manteiga),(pao),0.167,1.000,3.000,0.111,inf
12,"(leite, manteiga)",(pao),0.167,1.000,3.000,0.111,inf
3,(oleo),(feijao),0.250,1.000,2.000,0.125,inf
10,"(arroz, oleo)",(feijao),0.250,1.000,2.000,0.125,inf
11,(oleo),"(feijao, arroz)",0.250,1.000,2.000,0.125,inf
0,(feijao),(arroz),0.500,1.000,1.714,0.208,inf
1,(arroz),(feijao),0.500,0.857,1.714,0.208,3.5
2,(oleo),(arroz),0.250,1.000,1.714,0.104,inf


### 1. *Leverage* (alavancagem)

O *leverage* mede a diferença entre a frequência observada de ocorrência conjunta de $A$ e $B$ e a frequência que seria esperada caso os dois eventos fossem independentes.

#### Fórmula

$$\operatorname{Leverage}(A\rightarrow B)=P(A\cap B)-P(A)P(B).$$

O termo $P(A)P(B)$ representa a probabilidade esperada de $A$ e $B$ ocorrerem juntos sob independência. Já $P(A\cap B)$ é a proporção realmente observada de transações que contêm os dois.

#### Interpretação

- **Leverage = 0:** $A$ e $B$ comportam-se como eventos independentes.
- **Leverage > 0:** há associação positiva; $A$ e $B$ aparecem juntos mais frequentemente do que seria esperado sob independência.
- **Leverage < 0:** há associação negativa; $A$ e $B$ aparecem juntos menos frequentemente do que seria esperado sob independência.

#### Exemplo rápido

Considere a regra $\{\text{café}\}\rightarrow\{\text{açúcar}\}$. Se o *leverage* for 0,05, café e açúcar aparecem juntos em **5 pontos percentuais das transações a mais** do que seria esperado se suas compras fossem independentes. O *leverage*, portanto, mede uma **diferença absoluta de frequência**. Dizer “5 pontos percentuais” evita confundi-lo com um aumento relativo de 5%.

Como a fórmula usa a interseção e o produto das probabilidades marginais, trocar antecedente e consequente não altera o resultado:

$$\operatorname{Leverage}(A\rightarrow B)=\operatorname{Leverage}(B\rightarrow A).$$

### 2. *Conviction* (convicção)

A *conviction* avalia a força **direcional** da regra $A\rightarrow B$, concentrando-se nas situações em que ela falha:

$$A\cap\neg B.$$

Esses são os **contraexemplos** ou **violações da regra**: casos em que $A$ ocorre, mas $B$ não ocorre.

#### Fórmula

$$\operatorname{Conviction}(A\rightarrow B)=\frac{1-P(B)}{1-\operatorname{Confiança}(A\rightarrow B)}.$$

Como $\operatorname{Confiança}(A\rightarrow B)=P(B\mid A)$, também podemos escrever:

$$\operatorname{Conviction}(A\rightarrow B)=\frac{P(\neg B)}{P(\neg B\mid A)}.$$

Essa segunda forma evidencia o significado da medida: o numerador representa a ausência de $B$ em geral; o denominador representa a ausência de $B$ especificamente quando $A$ ocorre. Portanto, a *conviction* compara a taxa de violações esperada sob independência com a taxa realmente observada entre as transações que contêm $A$.

#### Interpretação

- **Conviction = 1:** a taxa de violações coincide com o que seria esperado sob independência.
- **Conviction > 1:** há associação positiva; quanto maior o valor, menos frequentemente $A\rightarrow B$ é violada.
- **Conviction < 1:** há associação negativa; a regra é violada mais frequentemente do que seria esperado.
- **Conviction = $\infty$:** a confiança é 1 e não há casos de $A$ sem $B$ na amostra.

No último caso, $P(B\mid A)=1$ e, consequentemente, $P(\neg B\mid A)=0$. A divisão por zero produz o valor infinito. Isso descreve os dados observados, mas não garante que nunca surgirá um contraexemplo em novas transações.

#### Exemplo rápido

Se $\operatorname{Conviction}(A\rightarrow B)=1{,}5$, a taxa de ausência de $B$ quando $A$ ocorre corresponde a $1/1{,}5\approx0{,}667$ da taxa-base de ausência de $B$. Em outras palavras, a ausência de $B$ é cerca de 1,5 vez menos frequente — ou 33,3% menor — quando $A$ ocorre. Quanto maior a *conviction*, menor a quantidade relativa de exceções à regra.

### Resumo comparativo

| Métrica | O que mede | Valor de independência | Interpretação principal |
|---|---|---:|---|
| **Leverage** | Diferença entre a ocorrência conjunta observada e a esperada sob independência | **0** | Quanto $A$ e $B$ aparecem juntos além ou aquém do esperado |
| **Conviction** | Violações esperadas da regra em comparação com as violações observadas | **1** | Força direcional da regra e quantidade relativa de exceções |

O *leverage* responde principalmente:

> **$A$ e $B$ aparecem juntos mais vezes do que seria esperado sob independência?**

A *conviction* responde:

> **Quando $A$ ocorre, quão rara é a violação da regra, isto é, $A$ ocorrer sem $B$?**

Não existe um limiar universal que torne uma regra relevante. As duas métricas devem ser analisadas com suporte, confiança, contagens, estabilidade em novos dados e conhecimento do domínio.


## Exemplo aplicado: da fórmula às tabelas

A seguir, selecionaremos três regras já geradas e transformaremos probabilidades em contagens. O conjunto possui $N=12$ transações. Para cada regra $A\rightarrow B$, a primeira tabela será construída a partir da fórmula do *leverage*:

$$\operatorname{Leverage}(A\rightarrow B)=P(A\cap B)-P(A)P(B).$$

| Coluna da Tabela 1 | Cálculo | Significado |
|---|---|---|
| <code>coocorrencias_observadas</code> | $N\,P(A\cap B)$ | número de transações em que $A$ e $B$ realmente aparecem juntos |
| <code>coocorrencias_esperadas</code> | $N\,P(A)P(B)$ | número esperado de coocorrências se $A$ e $B$ fossem independentes |
| <code>leverage</code> | $P(A\cap B)-P(A)P(B)$ | diferença entre as proporções observada e esperada |
| <code>excesso_de_coocorrencias</code> | $N\times\operatorname{Leverage}$ | diferença entre as contagens observada e esperada |

Portanto,

$$\text{coocorrências observadas}-\text{coocorrências esperadas}=N\times\operatorname{Leverage}.$$

A segunda tabela partirá da fórmula da *conviction*:

$$\operatorname{Conviction}(A\rightarrow B)=\frac{P(\neg B)}{P(\neg B\mid A)}.$$

| Coluna da Tabela 2 | Cálculo | Significado |
|---|---|---|
| <code>ocorrencias_antecedente</code> | $N\,P(A)$ | número de transações que contêm $A$ |
| <code>falhas_observadas</code> | $N\,P(A)[1-P(B\mid A)]$ | transações observadas com $A$ e sem $B$ |
| <code>falhas_esperadas</code> | $N\,P(A)[1-P(B)]$ | falhas esperadas se $A$ e $B$ fossem independentes |
| <code>confidence</code> | $P(B\mid A)$ | proporção das ocorrências de $A$ acompanhadas por $B$ |
| <code>conviction</code> | $P(\neg B)/P(\neg B\mid A)$ | razão entre a taxa esperada e a taxa observada de falhas |

Como o mesmo número de ocorrências de $A$ multiplica as duas taxas de falha, a *conviction* também pode ser obtida pela razão entre as contagens:

$$\operatorname{Conviction}(A\rightarrow B)=\frac{\text{falhas esperadas}}{\text{falhas observadas}}.$$

As coocorrências e falhas **esperadas** podem ser fracionárias, pois são valores teóricos sob independência. Já as contagens observadas são inteiras, ainda que a tabela as apresente com casas decimais.


In [3]:
def selecionar_regra(antecedente, consequente):
    mascara = (
        regras["antecedents"].eq(frozenset(antecedente))
        & regras["consequents"].eq(frozenset(consequente))
    )
    return regras.loc[mascara].copy()


casos = pd.concat([
    selecionar_regra({"pao", "leite"}, {"manteiga"}),
    selecionar_regra({"arroz"}, {"feijao"}),
    selecionar_regra({"feijao"}, {"arroz"}),
], ignore_index=True)
assert len(casos) == 3

N = len(transacoes)
casos["regra"] = casos.apply(
    lambda r: "{" + ", ".join(sorted(r["antecedents"])) + "} → {"
    + ", ".join(sorted(r["consequents"])) + "}", axis=1,
)
casos["coocorrencias_observadas"] = N * casos["support"]
casos["coocorrencias_esperadas"] = (
    N * casos["antecedent support"] * casos["consequent support"]
)
casos["excesso_de_coocorrencias"] = N * casos["leverage"]
casos["ocorrencias_antecedente"] = N * casos["antecedent support"]
casos["falhas_observadas"] = (
    casos["ocorrencias_antecedente"] * (1 - casos["confidence"])
)
casos["falhas_esperadas"] = (
    casos["ocorrencias_antecedente"] * (1 - casos["consequent support"])
)

tabela_leverage = casos[[
    "regra", "coocorrencias_observadas", "coocorrencias_esperadas",
    "leverage", "excesso_de_coocorrencias",
]].round(3)
tabela_conviction = casos[[
    "regra", "ocorrencias_antecedente", "falhas_observadas",
    "falhas_esperadas", "confidence", "conviction",
]].round(3)

print("Tabela 1 — Coocorrências e leverage")
display(tabela_leverage)
print("Tabela 2 — Violações da regra e conviction")
display(tabela_conviction)


Tabela 1 — Coocorrências e leverage


,regra,coocorrencias_observadas,coocorrencias_esperadas,leverage,excesso_de_coocorrencias
0,"{leite, pao} → {manteiga}",2.0,0.5,0.125,1.5
1,{arroz} → {feijao},6.0,3.5,0.208,2.5
2,{feijao} → {arroz},6.0,3.5,0.208,2.5


Tabela 2 — Violações da regra e conviction


,regra,ocorrencias_antecedente,falhas_observadas,falhas_esperadas,confidence,conviction
0,"{leite, pao} → {manteiga}",3.0,1.0,2.5,0.667,2.5
1,{arroz} → {feijao},7.0,1.0,3.5,0.857,3.5
2,{feijao} → {arroz},6.0,0.0,2.5,1.000,inf


### Leitura passo a passo das três regras

#### $\{pao, leite\}\rightarrow\{manteiga\}$

Na linha correspondente das tabelas, $A=\{pao, leite\}$ e $B=\{manteiga\}$. O antecedente aparece em 3 transações, o consequente em 2 e os três itens juntos em 2. Logo,

$$P(A)=\frac{3}{12},\qquad P(B)=\frac{2}{12},\qquad P(A\cap B)=\frac{2}{12}.$$

**Tabela 1 — *Leverage*.** Substituindo esses valores na fórmula:

$$\operatorname{Leverage}=\frac{2}{12}-\left(\frac{3}{12}\times\frac{2}{12}\right)=0{,}125.$$

A parcela entre parênteses é a proporção esperada sob independência. Em contagens, a tabela mostra $12\times(3/12)(2/12)=0{,}5$ coocorrência esperada e 2 observadas. A diferença é $2-0{,}5=1{,}5$, equivalente a $12\times0{,}125$. Portanto, os itens aparecem juntos em **12,5 pontos percentuais das transações a mais** do que seria esperado sob independência.

**Tabela 2 — *Conviction*.** A confiança é $2/3$, pois duas das três ocorrências do antecedente incluem manteiga. Assim,

$$P(\neg B)=1-\frac{2}{12}=\frac{10}{12},\qquad P(\neg B\mid A)=1-\frac{2}{3}=\frac{1}{3},$$

$$\operatorname{Conviction}=\frac{10/12}{1/3}=2{,}5.$$

A mesma conta aparece em forma de contagem: há 1 falha observada e $3\times(10/12)=2{,}5$ falhas esperadas; portanto, $2{,}5/1=2{,}5$. A taxa observada de violações é $1/2{,}5=0{,}4$ da esperada, isto é, 60% menor.

#### $\{arroz\}\rightarrow\{feijao\}$

Nesta regra, $A=\{arroz\}$ aparece em 7 transações, $B=\{feijao\}$ em 6 e ambos aparecem juntos em 6:

$$P(A)=\frac{7}{12},\qquad P(B)=\frac{6}{12},\qquad P(A\cap B)=\frac{6}{12}.$$

**Tabela 1 — *Leverage*.** Aplicando a fórmula:

$$\operatorname{Leverage}=\frac{6}{12}-\left(\frac{7}{12}\times\frac{6}{12}\right)=\frac{2{,}5}{12}\approx0{,}208.$$

A tabela apresenta 6 coocorrências observadas e $12\times(7/12)(6/12)=3{,}5$ esperadas. O excesso é $6-3{,}5=2{,}5$ transações, equivalente a **20,8 pontos percentuais acima da independência**.

**Tabela 2 — *Conviction*.** A confiança é $6/7$, pois uma das sete transações com arroz não contém feijão. Portanto,

$$P(\neg B)=1-\frac{6}{12}=\frac{1}{2},\qquad P(\neg B\mid A)=1-\frac{6}{7}=\frac{1}{7},$$

$$\operatorname{Conviction}=\frac{1/2}{1/7}=3{,}5.$$

Em contagens, a tabela mostra 1 falha observada e $7\times(1/2)=3{,}5$ esperadas. A razão $3{,}5/1=3{,}5$ produz o mesmo resultado; a taxa observada de violações é aproximadamente 71,4% menor que a esperada.

#### $\{feijao\}\rightarrow\{arroz\}$

Agora $A=\{feijao\}$ aparece em 6 transações, $B=\{arroz\}$ em 7 e ambos aparecem juntos em 6:

$$P(A)=\frac{6}{12},\qquad P(B)=\frac{7}{12},\qquad P(A\cap B)=\frac{6}{12}.$$

**Tabela 1 — *Leverage*.** A substituição na fórmula resulta em

$$\operatorname{Leverage}=\frac{6}{12}-\left(\frac{6}{12}\times\frac{7}{12}\right)=\frac{2{,}5}{12}\approx0{,}208.$$

Por isso, a Tabela 1 repete as 6 coocorrências observadas, 3,5 esperadas e o excesso de 2,5 da regra inversa: o *leverage* é simétrico.

**Tabela 2 — *Conviction*.** Todas as seis transações com feijão contêm arroz, de modo que a confiança é $6/6=1$:

$$P(\neg B)=1-\frac{7}{12}=\frac{5}{12},\qquad P(\neg B\mid A)=1-1=0,$$

$$\operatorname{Conviction}=\frac{5/12}{0}=\infty.$$

Em contagens, seriam esperadas $6\times(5/12)=2{,}5$ falhas sob independência, mas foram observadas zero. Dividir 2,5 por zero também produz infinito. Isso significa **ausência de contraexemplos nesta amostra de 12 transações**, não uma regra universal.

As regras arroz→feijão e feijão→arroz têm o mesmo suporte e o mesmo *leverage* — aproximadamente 0,208 — porque o *leverage* é simétrico. Suas *convictions* são diferentes — 3,5 e infinito — porque a *conviction* considera as violações em uma direção específica.


## Avaliação responsável

Uma análise defensável combina força descritiva, contagens, estabilidade em outros períodos, novidade, redundância, acionabilidade e riscos. Promoções, disponibilidade, sazonalidade e segmentação podem explicar associações. Testar muitas regras aumenta coincidências; um período de confirmação e busca controlada reduzem conclusões oportunistas. Efeito de intervenção exige desenho causal apropriado.


> **U04-NB03-V01 — Verifique seu entendimento:** por que *leverage* 0,125 deve ser interpretado como 12,5 pontos percentuais acima da independência, e não como aumento de 12,5% nas vendas?

> **U04-NB03-E01 — Exercício:** para a regra $\{oleo\}\rightarrow\{feijao\}$ exibida anteriormente, use as 12 transações para calcular $P(A)$, $P(B)$, $P(A\cap B)$, *leverage*, falhas observadas, falhas esperadas e *conviction*. Relacione cada resultado às colunas das duas tabelas.

> **U04-NB03-E02 — Atividade integradora:** escolha uma base transacional pequena; documente unidade, período e representação; minere regras com dois limiares; selecione no máximo cinco resultados usando suporte, confiança, *lift*, *leverage*, *conviction*, estabilidade e utilidade; e produza uma recomendação com contagens, limitações e ressalva causal.


## Síntese

- Regra forte por limiar não é automaticamente interessante ou acionável.
- Taxa-base, contagem, estabilidade e redundância complementam as métricas.
- *Leverage* mede excesso absoluto de coocorrência e é simétrico; *conviction* mede violações relativas e é direcional.
- Fórmulas, probabilidades e contagens devem conduzir à mesma interpretação.
- Regras observacionais sustentam hipóteses, não conclusões causais.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 4.
- GOOGLE. [Explicação complementar sobre *leverage* e *conviction*](https://share.google/aimode/vNmyzZllViNT8IJnu). Material indicado pelo docente.
